# NanoGPT scratchpad

In [23]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [7]:
# Download the raw tiny shakespeare dataset
!curl -O https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1089k  100 1089k    0     0  1074k      0  0:00:01  0:00:01 --:--:-- 1074k-  0:01:01 18133


In [8]:
# read to inspect
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [10]:
print(f"Length of dataset in characters: {len(text)}")

Length of dataset in characters: 1115394


In [11]:
# check out the first 1000 characters:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [12]:
# all the unique characters that occur iin this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [14]:
stoi = {ch:i+1 for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}
encode = lambda s: [stoi[c] for c in s] # encoder: takes a string, outputs a list of corresponding integers 
decode = lambda l: ''.join([itos[i] for i in l]) # encoder: takes a list of integers, outputs a string

# like this
print(encode("I'm GPU-poor"))
print(decode(encode("I'm GPU-poor")))


[22, 6, 52, 2, 20, 29, 34, 8, 55, 54, 54, 57]
I'm GPU-poor


In [15]:
# encode entire dataset and store as a torch.Tensor
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([19, 48, 57, 58, 59,  2, 16, 48, 59, 48, 65, 44, 53, 11,  1, 15, 44, 45,
        54, 57, 44,  2, 62, 44,  2, 55, 57, 54, 42, 44, 44, 43,  2, 40, 53, 64,
         2, 45, 60, 57, 59, 47, 44, 57,  7,  2, 47, 44, 40, 57,  2, 52, 44,  2,
        58, 55, 44, 40, 50,  9,  1,  1, 14, 51, 51, 11,  1, 32, 55, 44, 40, 50,
         7,  2, 58, 55, 44, 40, 50,  9,  1,  1, 19, 48, 57, 58, 59,  2, 16, 48,
        59, 48, 65, 44, 53, 11,  1, 38, 54, 60,  2, 40, 57, 44,  2, 40, 51, 51,
         2, 57, 44, 58, 54, 51, 61, 44, 43,  2, 57, 40, 59, 47, 44, 57,  2, 59,
        54,  2, 43, 48, 44,  2, 59, 47, 40, 53,  2, 59, 54,  2, 45, 40, 52, 48,
        58, 47, 13,  1,  1, 14, 51, 51, 11,  1, 31, 44, 58, 54, 51, 61, 44, 43,
         9,  2, 57, 44, 58, 54, 51, 61, 44, 43,  9,  1,  1, 19, 48, 57, 58, 59,
         2, 16, 48, 59, 48, 65, 44, 53, 11,  1, 19, 48, 57, 58, 59,  7,  2, 64,
        54, 60,  2, 50, 53, 54, 62,  2, 16, 40, 48, 60, 58,  2, 26, 40, 57, 42,
      

In [16]:
# train / val data split
n = int(0.9*len(data)) # first 90% will be train, remaining 10% will be validation set
train_data = data[:n]
val_data = data[n:]

In [17]:
context_length = 8
train_data[:context_length+1]

tensor([19, 48, 57, 58, 59,  2, 16, 48, 59])

In [ ]:
# show sliding window for extracting context_length individual training examples
x = train_data[:context_length]
y = train_data[1:context_length+1] # offset to the right by one position
for i, t in enumerate(range(context_length)):
    context = x[:t+1]
    target = y[t]
    print(f"{i+1}: when input is {context} the target is {target}")

1: when input is tensor([19]) the target is 48
2: when input is tensor([19, 48]) the target is 57
3: when input is tensor([19, 48, 57]) the target is 58
4: when input is tensor([19, 48, 57, 58]) the target is 59
5: when input is tensor([19, 48, 57, 58, 59]) the target is 2
6: when input is tensor([19, 48, 57, 58, 59,  2]) the target is 16
7: when input is tensor([19, 48, 57, 58, 59,  2, 16]) the target is 48
8: when input is tensor([19, 48, 57, 58, 59,  2, 16, 48]) the target is 59


In [21]:
torch.manual_seed(1337)
batch_size = 4 # no. independent sequences we will process in parallel
context_length = 8 # max context length of input for predictions

def get_batch(split):
    # generate a small batch of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - context_length, (batch_size, )) # random sample of batch_size indices from the data (offset by the max context length)
    x = torch.stack([data[i:i+context_length] for i in ix])
    y = torch.stack([data[i+1:i+context_length+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)


inputs:
torch.Size([4, 8])
tensor([[25, 44, 59,  6, 58,  2, 47, 44],
        [45, 54, 57,  2, 59, 47, 40, 59],
        [53, 59,  2, 59, 47, 40, 59,  2],
        [26, 18, 28, 11,  1, 22,  2, 55]])
targets:
torch.Size([4, 8])
tensor([[44, 59,  6, 58,  2, 47, 44, 40],
        [54, 57,  2, 59, 47, 40, 59,  2],
        [59,  2, 59, 47, 40, 59,  2, 47],
        [18, 28, 11,  1, 22,  2, 55, 40]])


In [22]:
print(xb) # input to the transformer

tensor([[25, 44, 59,  6, 58,  2, 47, 44],
        [45, 54, 57,  2, 59, 47, 40, 59],
        [53, 59,  2, 59, 47, 40, 59,  2],
        [26, 18, 28, 11,  1, 22,  2, 55]])


In [ ]:
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets):
        logits = self.token_embedding_table(idx) # shape (B, T, C) (batch, time, context length)
        return logits